In [1]:
from lib import read_chain_file
from src.readers import ChainReader, CoolerPolars, read_cooler, read_pairs, read_cooler_square, read_cooler_chunk
from src.transformers import remap_bins, save_cooler_chunks, save_cooler_separate
from src.pixel_dividers import LinearDivider, CVDNorm, PolynimialSplineDivider, Linear2DDivider
from src.readers.cooler_reader import _COOLER_BINS_SCHEMA

import cooler
from cooler.create import aggregate_records, create_cooler
from cooltools import expected_cis, expected_trans

import polars as pl
import bioframe


import matplotlib.pyplot as plt

import numpy as np
from scipy.interpolate import UnivariateSpline

from typing import Optional, List, Tuple, Dict, Any, Callable, Literal
from typing import Iterator

from scipy.stats import wasserstein_distance_nd

import time

/Users/egorpitikov/Documents/pets/liftover_2d/.venv/lib/python3.12/site-packages/numba/core/decorators.py:246: RuntimeWarning: nopython is set for njit and is ignored
  warnings.warn('nopython is set for njit and is ignored', RuntimeWarning)


In [2]:
test_params = [
    {
        'source_cooler': '/Users/egorpitikov/Documents/pets/liftover_2d/YueLab-muscle-HiC.danRer10.mapq_30.all_res.mcool',
        'target_cooler': '/Users/egorpitikov/Documents/pets/liftover_2d/YueLab-muscle-HiC.danRer11.mapq_30.all_res.mcool',
        'output_cooler': '/Users/egorpitikov/Documents/big_sasha_test/YueLab-muscle-HiC.model_{{model_type}}.danRer10_to_danRer11.res{resolution}.cool',
        'chain_file': '/Users/egorpitikov/Downloads/danRer10ToDanRer11.over.chain',
        'resolutions': [100_000, 50_000]
    },
    {
        'source_cooler': '/Users/egorpitikov/Documents/osfstorage/adult_CNS.dm3.mapq_30.100.mcool',
        'target_cooler': '/Users/egorpitikov/Documents/osfstorage/adult_CNS.dm6.mapq_30.100.mcool',
        'output_cooler': '/Users/egorpitikov/Documents/big_sasha_test/adult_CNS.model_{{model_type}}.dm6_to_dm3.res{resolution}.cool',
        'chain_file': '/Users/egorpitikov/Downloads/dm3ToDm6.over.chain',
        'resolutions': [100_000, 50_000]
    }
]

In [3]:
test_params = [
    {
        'source_cooler': '/Users/egorpitikov/Documents/osfstorage/adult_CNS.dm3.mapq_30.100.mcool',
        'target_cooler': '/Users/egorpitikov/Documents/osfstorage/adult_CNS.dm6.mapq_30.100.mcool',
        'output_cooler': '/Users/egorpitikov/Documents/big_sasha_test/adult_CNS.model_{{model_type}}.dm6_to_dm3.res{resolution}.cool',
        'chain_file': '/Users/egorpitikov/Downloads/dm3ToDm6.over.chain',
        'resolutions': [10_000]
    }
]

In [4]:
to_test_all = []

for p in test_params:
    for res in p['resolutions']:
        to_ans = p.copy()
        to_ans['resolution'] = res 
        del to_ans['resolutions']
        to_test_all.append(to_ans)

In [5]:
def cals_cvd_coeffs(clr_obj, nproc=4):
    view_df = bioframe.make_viewframe(clr_obj.chromsizes)
    cis = expected_cis(
        clr=clr_obj,
        view_df=view_df,
        smooth=True,
        aggregate_smoothed=True,
        nproc=nproc,
        clr_weight_name=None
    )
    cis_gr = cis.groupby(['region1']).get_group(view_df.chrom[0])
    cvd_cis = cis_gr.set_index('dist')['balanced.avg.smoothed.agg']


    spline = UnivariateSpline(
        np.log10(cvd_cis.index[2:]), 
        np.log10(cvd_cis.values[2:]), 
        k=5
    )

    cvd_cis[0] = spline(0)
    cvd_cis[1] = spline(1)

    fig, ax = plt.subplots(figsize=(10, 5))
    ax.plot(cvd_cis.index, cvd_cis.values, label='CVD')
    ax.plot(cvd_cis.index, spline(cvd_cis.index), label='Spline', linestyle='--')
    ax.legend()
    plt.show()


    trans = expected_trans(
        clr=clr_obj,
        view_df=view_df,
        nproc=nproc,
        clr_weight_name=None
    )
    cvd_trans = trans.set_index(['region1', 'region2'])['balanced.avg'].mean()
    return cvd_cis, cvd_trans


import numpy as np
import matplotlib.pyplot as plt
from scipy.interpolate import UnivariateSpline
import pandas as pd

def cals_cvd_coeffs_loglog(clr_obj, nproc=4, s_factor=None, k=3):
    """
    Подсчёт CVD и TAD 'cis' и 'trans' с лог‑лог сплайном для 'cis'.
    Параметр s_factor (если None) равен len(log_d)*Var(log_y) (вроде подсказки).
    """
    view_df = bioframe.make_viewframe(clr_obj.chromsizes)

    cis = expected_cis(
        clr=clr_obj,
        view_df=view_df,
        smooth=True,
        aggregate_smoothed=True,
        nproc=nproc,
        clr_weight_name=None,
        ignore_diags=0
    )
    chrom = view_df.chrom[0]
    cis_chrom = cis.groupby(['region1']).get_group(chrom)

   # display(cis_chrom)

    c = cis_chrom.set_index('dist')['count.avg.smoothed.agg'].sort_index()

#    fig, ax = plt.subplots(figsize=(10, 5))
#    ax.plot(c.index, c.values, label='CVD')
#    ax.plot(c.index, spline(c.index), label='Spline', linestyle='--')
#    ax.legend()
#    plt.show()

    valid_mask = (c.index > 1) & (c.values > 0)
    d = c.index[valid_mask]  # distances > 0
    y = c.values[valid_mask] # contacts > 0

    X = np.log(d)              # log(dist)
    Y = np.log(y)              # log(contact)
    if s_factor is None:
        s_factor = len(X) * np.var(Y)

    spline = UnivariateSpline(X, Y, k=k, s=s_factor)

    ff = pd.Series(
        np.exp(spline(X)),
        index=d,
        name='cvd_fitted'
    )

    if 1 in c.index:
        ff_1 = spline(np.log(1))
        ff.loc[1] = np.exp(ff_1)

    trans = expected_trans(
        clr=clr_obj,
        view_df=view_df,
        nproc=nproc,
        clr_weight_name=None
    )

    cvd_trans = trans.set_index(['region1', 'region2'])['count.avg'].mean()

    return c, ff, cvd_trans



In [6]:
def run_test(
    source_cooler_path: str,
    target_cooler_path: str,
    chain_path: str,
    resolution: int,
    path_for_answer:str
):
    source_cooler = cooler.Cooler(f'{source_cooler_path}::resolutions/{resolution}')
    target_cooler = cooler.Cooler(f'{target_cooler_path}::resolutions/{resolution}')
    chain_obj = ChainReader(open(chain_path, 'rb'))

    print(f'Starts for {source_cooler_path} in {resolution}')

    bins_sour = pl.from_pandas(
        source_cooler.bins()[:].reset_index(names='bin_id').loc[:, ['bin_id', 'chrom', 'start', 'end']]
    ).cast(_COOLER_BINS_SCHEMA)

    bins_tar = pl.from_pandas(
        target_cooler.bins()[:].reset_index(names='bin_id').loc[:, ['bin_id', 'chrom', 'start', 'end']]
    ).cast(_COOLER_BINS_SCHEMA)

    remap_schema = remap_bins(
        source=bins_sour,
        target=bins_tar,
        chains=chain_obj
    )

    s, cis_norm, trans_norm = cals_cvd_coeffs_loglog(source_cooler, nproc=4, k=4)


    mtt = dict(
       # modelka_linear = LinearDivider(mode='resample'),
       # mdelka_cvd = CVDNorm(mode='resample', cis=cis_norm, trans=trans_norm),
        modelka_spline = PolynimialSplineDivider(mode='resample', k=4, ident=5, scale_factor=5)
    )


    def predictalka(cooler_file, modelka, remap_schema, traget_bins):
        for cur_bbox, hic_part in read_cooler_square(cooler_file=cooler_file, square_size=800, square_overlap=5):
            if hic_part.counts.shape[0] > 0:
                preds = modelka.predict(hic_part, remap_schema=remap_schema)
                min_1_bin, max_1_bin, min_2_bin, max_2_bin = cur_bbox
                if min_1_bin != 0:
                    min_1_bin += 5
                if max_1_bin != cooler_file.info['nbins']:
                    max_1_bin -= 6
                if min_2_bin != 0:
                    min_2_bin += 5      
                if max_2_bin != cooler_file.info['nbins']:
                    max_2_bin -= 6
                yield preds.filter(
                    pl.col('bin1_id').is_between(min_1_bin, max_1_bin) 
                    & pl.col('bin2_id').is_between(min_2_bin, max_2_bin)
                )
            else:
                continue

    res_clr = {}
    top_c = 0

    models_dict = {}
    #save_cooler_chunks(cooler_polars_iterator: Iterable[pl.DataFrame], res_path: str, bins: pl.DataFrame, pixels: pl.DataFrame)
    for model_type, modelka in mtt.items():
        print('Running model:', model_type)
        
        time_start = time.perf_counter()
        res_clr[model_type] = save_cooler_chunks(
            pixels_iterator=predictalka(
                cooler_file=source_cooler,
                modelka=modelka,
                remap_schema=remap_schema,
                traget_bins=bins_tar
            ),
            bins=bins_tar.select(
                pl.col('chrom'),
                pl.col('start').cast(pl.Int32).alias('start'),
                pl.col('end').cast(pl.Int32).alias('end'),
            ).to_pandas(),
            res_path=path_for_answer.format(model_type=model_type),
        # ttl_seconds=1800,
        # print_progress=True
        )
        time_end = time.perf_counter()
        print('Saved cooler to:', path_for_answer.format(model_type=model_type))
        top_c += 1
        models_dict[model_type] = time_end - time_start
    return models_dict

In [7]:
to_test_all

[{'source_cooler': '/Users/egorpitikov/Documents/osfstorage/adult_CNS.dm3.mapq_30.100.mcool',
  'target_cooler': '/Users/egorpitikov/Documents/osfstorage/adult_CNS.dm6.mapq_30.100.mcool',
  'output_cooler': '/Users/egorpitikov/Documents/big_sasha_test/adult_CNS.model_{{model_type}}.dm6_to_dm3.res{resolution}.cool',
  'chain_file': '/Users/egorpitikov/Downloads/dm3ToDm6.over.chain',
  'resolution': 10000}]

In [8]:
for params in to_test_all:
    models_dict = run_test(
        source_cooler_path=params['source_cooler'],
        target_cooler_path=params['target_cooler'],
        chain_path=params['chain_file'],
        resolution=params['resolution'],
        path_for_answer=params['output_cooler'].format(resolution=params['resolution'])
    )
    params['timings'] = models_dict


Starts for /Users/egorpitikov/Documents/osfstorage/adult_CNS.dm3.mapq_30.100.mcool in 10000


/Users/egorpitikov/Documents/pets/liftover_2d/.venv/lib/python3.12/site-packages/multiprocess/popen_fork.py:66: RuntimeWarning: Using fork() can cause Polars to deadlock in the child process.
In addition, using fork() with Python in general is a recipe for mysterious
deadlocks and crashes.

The most likely reason you are seeing this error is because you are using the
multiprocessing module on Linux, which uses fork() by default. This will be
fixed in Python 3.14. Until then, you want to use the "spawn" context instead.

See https://docs.pola.rs/user-guide/misc/multiprocessing/ for details.

If you really know what your doing, you can silence this warning with the warning module
or by setting POLARS_ALLOW_FORKING_THREAD=1.

  self.pid = os.fork()


Running model: modelka_spline
Saved cooler to: /Users/egorpitikov/Documents/big_sasha_test/adult_CNS.model_modelka_spline.dm6_to_dm3.res10000.cool
